# 🚀 EchoVision: Cloud Multi-Modal Video QA Framework
### ⚡ Kaggle Notebook T4 GPU Edition

**EchoVision** is an advanced multi-modal Retrieval-Augmented Generation (RAG) framework for Video Question Answering (Video QA) optimized to run on **Kaggle Notebooks with GPU T4 x 1**.

| Model | Role & Use in EchoVision |
| :--- | :--- |
| **GPT-4o-mini** | Structured keyframe captioning, query modality classification, & answer generation |
| **BGE-large-en-v1.5** | Dense text embeddings for cross-modal vector indexing & retrieval |
| **BM25** | Keyword-based lexical re-ranking across extracted audio/visual facts |
| **PySceneDetect** | Content-aware adaptive video scene detection |
| **CLIP** | Importance-aware visual keyframe selection & semantic deduplication |
| **GroundingDINO** | Open-vocabulary text-guided object detection |
| **RT-DETR** | High-throughput real-time object detection |
| **SimpleSort** | Multi-frame Kalman/IoU object tracking, quadrant & trajectory estimation |
| **Whisper large-v3-turbo** | High-accuracy timestamped speech transcription |
| **AST** | Audio Spectrogram Transformer for acoustic event classification & spatial balance |

---
### ⚙️ Required Kaggle Settings (Right Sidebar Panel):
1. **Accelerator**: Select **GPU T4 x 1** (or P100 GPU).
2. **Internet**: Toggle **Internet ON** (Required to download HuggingFace models & pip packages).
3. **Secrets (Optional)**: If using GPT-4o-mini, add your `OPENAI_API_KEY` under **Add-ons** $\rightarrow$ **Secrets**.


## 1️⃣ Hardware & Environment Verification
Verify that Kaggle has assigned the Tesla T4 GPU and Internet connectivity is active.


In [ ]:
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM      : {vram_gb:.2f} GB")
    print("[✓] Tesla T4 GPU confirmed!")
else:
    print("[!] Please turn on GPU in the Kaggle notebook settings panel on the right.")
print("=" * 60)


## 2️⃣ Configure OpenAI API Key (Optional for GPT-4o-mini)
On Kaggle, you can retrieve your `OPENAI_API_KEY` securely from Kaggle Secrets, or enter it manually below.


In [ ]:
import os, getpass

api_key = None
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("OPENAI_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    api_key = getpass.getpass("Enter your OPENAI_API_KEY (optional, press Enter to use local models): ").strip()

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key
    os.environ["GPT_MODEL"] = "gpt-4o-mini"
    print("[✓] OpenAI API Key configured successfully! GPT-4o-mini is active.")
else:
    print("[i] No OpenAI API Key provided. Pipeline will use local offline HuggingFace models.")


## 3️⃣ Setup Workspace in `/kaggle/working`


In [ ]:
import os, sys

WORKING_DIR = "/kaggle/working"
os.chdir(WORKING_DIR)

# If repository is cloned from GitHub into Kaggle (uncomment if applicable):
# !git clone https://github.com/YOUR_USERNAME/cloud_EVL.git
# os.chdir("/kaggle/working/cloud_EVL")

sys.path.append(os.getcwd())
print("Working Directory:", os.getcwd())
!ls -lh


## 4️⃣ Install Multi-Modal Dependencies
Install `ffmpeg`, `libsndfile1`, and all required multi-modal packages:


In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!pip install -q -r requirements.txt

import faster_whisper
import transformers
import scenedetect
import chromadb
import faiss
import rank_bm25
import timm
import openai
print("\n[✓] All dependencies installed successfully on Kaggle!")


## 5️⃣ Configure Model Parameters & T4 Memory Strategy


In [ ]:
import os

os.environ["WHISPER_MODEL"] = "large-v3-turbo"
os.environ["AST_MODEL"] = "MIT/ast-finetuned-audioset-10-10-0.4593"
os.environ["TEXT_EMBEDDING_MODEL"] = "BAAI/bge-large-en-v1.5"
os.environ["RTDETR_MODEL"] = "PekingU/rtdetr_r50vd"
os.environ["GROUNDING_DINO_MODEL"] = "IDEA-Research/grounding-dino-tiny"
os.environ["ENABLE_BM25"] = "1"
os.environ["BM25_WEIGHT"] = "0.4"
os.environ["CONCURRENT_INGESTION"] = "0"  # Memory-safe sequential mode for T4

import config
print("=" * 60)
print(f"Device                 : {config.DEVICE}")
print(f"Torch Precision        : {config.TORCH_DTYPE}")
print(f"Speech Transcription   : Whisper {config.WHISPER_MODEL}")
print(f"Audio Event Model      : {config.AST_MODEL}")
print(f"Text Embeddings        : {config.TEXT_EMBEDDING_MODEL}")
print(f"Object Detector        : RT-DETR + GroundingDINO")
print(f"Object Tracker         : SimpleSort (Kalman/IoU)")
print(f"BM25 Keyword Re-ranker : {'Enabled (Weight=' + str(config.BM25_WEIGHT) + ')' if config.ENABLE_BM25 else 'Disabled'}")
print(f"Ingestion Pipeline     : Sequential (Memory-Safe for Tesla T4)")
print("=" * 60)


## 6️⃣ Discover Videos & Inspect Questions Dataset


In [ ]:
from ingestion import discover_videos
import json

videos = discover_videos("smoketest/videos")
print(f"Discovered {len(videos)} video(s) in smoketest/videos:")
for v in videos[:6]:
    print(f"  - {os.path.basename(v)}")

json_file = "smoketest/json/smoketest_questions.json"
if os.path.exists(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        q_data = json.load(f)
    print(f"\nLoaded {len(q_data)} questions from {json_file}")


## 7️⃣ Run Stage 1 (Sequential Extraction on T4 GPU)


In [ ]:
!python ingestion.py --dataset_dir smoketest --sequential


## 8️⃣ Run Stage 2 & 3 (Decoupled Retrieval & Answer Generation)


In [ ]:
!python main.py --dataset_dir smoketest --output_dir output


## 9️⃣ View Evaluation Metrics & Sample Predictions


In [ ]:
import os, json
import pandas as pd

eval_path = "output/evaluation_summary.json"
if os.path.exists(eval_path):
    with open(eval_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    print("=" * 60)
    print("                 EVALUATION METRICS REPORT                  ")
    print("=" * 60)
    metrics = eval_data.get("metrics", {})
    print(f"Total Evaluated Questions    : {metrics.get('total_evaluated', 0)}")
    print(f"Exact Match Accuracy         : {metrics.get('exact_match_accuracy', 0)}%")
    print(f"Relaxed Substring Accuracy   : {metrics.get('relaxed_accuracy', 0)}%")
    if metrics.get("category_breakdown"):
        print("\nCategory Breakdown:")
        for cat, c_res in metrics["category_breakdown"].items():
            print(f"  - {cat:<20}: {c_res['accuracy']:>6.2f}% ({c_res['correct']}/{c_res['total']})")
    print("=" * 60)

pred_path = "output/smoketest_questions.json"
if os.path.exists(pred_path):
    with open(pred_path, "r", encoding="utf-8") as f:
        preds = json.load(f)
    df = pd.DataFrame(preds)
    cols = [c for c in ["video_id", "category", "question", "predicted_answer", "ground_truth_answer"] if c in df.columns]
    display(df[cols].head(15))


## 🔟 Run Scientific Ablation Experiments (`run_ablations.py`)


In [ ]:
!python run_ablations.py --dataset_dir smoketest --output_dir output/ablations


## 1️⃣1️⃣ Interactive Single-Video Query Demo


In [ ]:
import os
from ingestion import discover_videos, Stage1Ingestor
from main import answer_question_for_video
from stage2_online.question_classifier import QuestionClassifier
from stage2_online.deduplicator import Deduplicator
from stage2_online.reranker import ReRanker
from stage2_online.sufficiency_gate import SufficiencyGate
from stage3_generator.generator import Generator

sample_videos = discover_videos("smoketest/videos")
if sample_videos:
    test_video = sample_videos[0]
    test_question = "what is behind the person in purple clothes?"
    
    print(f"Selected Video: {test_video}")
    print(f"Question      : {test_question}\n")
    
    ingestor = Stage1Ingestor()
    indexer, reused, err = ingestor.process_single_video(test_video)
    
    if indexer:
        shared_components = {
            'qc': QuestionClassifier(),
            'dedup': Deduplicator(),
            'reranker': ReRanker(),
            'gate': SufficiencyGate(),
            'generator': Generator()
        }
        answer = answer_question_for_video(indexer, test_question, shared_components)
        print("\n" + "=" * 50)
        print(f"PREDICTED ANSWER: {answer}")
        print("=" * 50)
